# MOEX Portfolio Optimizer

Автоматический подбор оптимального инвестиционного портфеля акций MOEX.

## Пайплайн
1. Загрузка данных с MOEX ISS API
2. Фильтрация по ликвидности и аномалиям
3. Расчёт матрицы корреляций
4. Построение графа корреляций и поиск максимальной клики
5. Markowitz Mean-Variance оптимизация
6. Black-Litterman и HRP
7. Monte Carlo симуляция и анализ рисков
8. Ребалансирование и стресс-тестирование
9. Визуализация результатов

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from moex_portfolio.config import (
    CORR_THRESHOLD, MIN_TURNOVER, MIN_OBSERVATIONS,
    START_DATE, END_DATE, MAX_WEIGHT, RISK_FREE_RATE,
    REBALANCE_FREQ_DAYS, TRANSACTION_COST_BPS,
)
from moex_portfolio.data_loader import load_all_data
from moex_portfolio.filters import prepare_returns
from moex_portfolio.correlation import compute_correlation_matrix, plot_correlation_heatmap
from moex_portfolio.graph_analysis import build_correlation_graph, find_max_clique
from moex_portfolio.visualization import (
    plot_full_graph, plot_clique_on_graph, plot_clique_separate,
    plot_clique_heatmap, plot_total_returns
)
from moex_portfolio.metrics import portfolio_metrics
from moex_portfolio.optimizer import max_sharpe_portfolio, min_variance_portfolio, efficient_frontier
from moex_portfolio.risk_models import covariance_matrix, compute_beta, compute_alpha
from moex_portfolio.analytics import (
    equity_curve, monte_carlo_simulation, var_historical, cvar_historical,
    rolling_correlation, rolling_beta,
)
from moex_portfolio.black_litterman import create_views_from_correlation, optimize_black_litterman
from moex_portfolio.hrp import optimize_hrp
from moex_portfolio.rebalancing import (
    RebalanceConfig, simulate_rebalancing, simulate_buy_and_hold, compare_strategies
)
from moex_portfolio.stress_test import run_all_scenarios, stress_results_to_dataframe
from moex_portfolio.profiles import save_profile, load_profile, list_profiles

print(f"Период данных: {START_DATE} — {END_DATE}")
print(f"Порог корреляции: {CORR_THRESHOLD}")
print(f"Минимальный оборот: {MIN_TURNOVER / 1_000_000:.0f}M RUB")

## 1. Загрузка данных

In [ ]:
raw_data = load_all_data(use_cache=True)
print(f"Загружено: {raw_data.shape[0]} дней, {raw_data.shape[1]} столбцов")

## 2. Фильтрация и расчёт доходностей

In [ ]:
returns, valid_tickers = prepare_returns(raw_data)
print(f"После фильтрации: {len(valid_tickers)} акций, {len(returns)} периодов")
returns.head()

## 3. Матрица корреляций

In [ ]:
corr = compute_correlation_matrix(returns)
fig = plot_correlation_heatmap(corr, title="Матрица корреляций акций MOEX")
plt.show()

## 4. Граф корреляций и максимальная клика

In [ ]:
G = build_correlation_graph(corr, threshold=CORR_THRESHOLD)
clique = find_max_clique(G)
print(f"Максимальная клика: {len(clique)} акций")
print(f"Акции: {', '.join(clique)}")

In [ ]:
fig = plot_full_graph(G, clique=clique)
plt.show()

In [ ]:
fig = plot_clique_separate(G, clique)
plt.show()

In [ ]:
fig = plot_clique_heatmap(returns, clique)
plt.show()

## 5. Доходности акций клики

In [ ]:
fig = plot_total_returns(returns, clique)
plt.show()

## 6. Оптимизация портфеля (Markowitz)

Используем Ledoit-Wolf сжатие ковариации для более стабильной оценки.

In [ ]:
clique_returns = returns[clique]
mean_ret = clique_returns.mean()
cov = covariance_matrix(clique_returns, method='ledoit_wolf')

print(f"Ковариационная матрица (Ledoit-Wolf): {cov.shape}")

opt = max_sharpe_portfolio(mean_ret, cov)
print("\n=== Максимальный Sharpe Ratio ===")
print(f"Доходность: {opt['return']:.2%}")
print(f"Волатильность: {opt['volatility']:.2%}")
print(f"Sharpe: {opt['sharpe']:.3f}")
print()
for t, w in sorted(zip(clique, opt['weights']), key=lambda x: -x[1]):
    print(f"  {t}: {w:.2%}")

In [ ]:
min_var = min_variance_portfolio(mean_ret, cov)
print("=== Минимальная волатильность ===")
print(f"Доходность: {min_var['return']:.2%}")
print(f"Волатильность: {min_var['volatility']:.2%}")
print(f"Sharpe: {min_var['sharpe']:.3f}")
print()
for t, w in sorted(zip(clique, min_var['weights']), key=lambda x: -x[1]):
    print(f"  {t}: {w:.2%}")

## 7. Эффективный фронтер

In [ ]:
ef = efficient_frontier(mean_ret, cov, n_points=50)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(ef['volatility'], ef['return'], c=ef['sharpe'], cmap='viridis', s=10)
ax.scatter(opt['volatility'], opt['return'], marker='*', s=300, c='red', label='Max Sharpe', zorder=5)
ax.scatter(min_var['volatility'], min_var['return'], marker='*', s=300, c='blue', label='Min Variance', zorder=5)
ax.set_xlabel('Волатильность (годовая)')
ax.set_ylabel('Доходность (годовая)')
ax.set_title('Эффективный фронтер')
ax.legend()
plt.colorbar(ax.collections[0], ax=ax, label='Sharpe Ratio')
plt.tight_layout()
plt.show()

## 8. Black-Litterman модель

In [ ]:
P, Q = create_views_from_correlation(clique_returns, corr.loc[clique, clique], top_n=5)
bl_result = optimize_black_litterman(clique_returns, P, Q, max_weight=MAX_WEIGHT)

print("=== Black-Litterman ===")
print(f"Доходность: {bl_result['return']:.2%}")
print(f"Волатильность: {bl_result['volatility']:.2%}")
print(f"Sharpe: {bl_result['sharpe']:.3f}")
print()
for t, w in sorted(zip(clique, bl_result['weights']), key=lambda x: -x[1]):
    print(f"  {t}: {w:.2%}")

## 9. Hierarchical Risk Parity (HRP)

In [ ]:
hrp_result = optimize_hrp(clique_returns, max_weight=MAX_WEIGHT)

print("=== HRP ===")
print(f"Доходность: {hrp_result['return']:.2%}")
print(f"Волатильность: {hrp_result['volatility']:.2%}")
print(f"Sharpe: {hrp_result['sharpe']:.3f}")
print()
for t, w in sorted(hrp_result['weights_dict'].items(), key=lambda x: -x[1]):
    print(f"  {t}: {w:.2%}")

In [ ]:
strategies = pd.DataFrame({
    'Strategy': ['Markowitz', 'Min Variance', 'Black-Litterman', 'HRP'],
    'Return': [opt['return'], min_var['return'], bl_result['return'], hrp_result['return']],
    'Volatility': [opt['volatility'], min_var['volatility'], bl_result['volatility'], hrp_result['volatility']],
    'Sharpe': [opt['sharpe'], min_var['sharpe'], bl_result['sharpe'], hrp_result['sharpe']],
})
print(strategies.to_string(index=False))

## 10. Кривая роста капитала

In [ ]:
eq = equity_curve(clique_returns, opt['weights'])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(eq.index, eq.values, linewidth=1.5, color='#1f77b4')
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Кривая роста капитала (Max Sharpe Portfolio)')
ax.set_ylabel('Накопленная доходность')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Monte Carlo симуляция

In [ ]:
mc = monte_carlo_simulation(mean_ret, cov, opt['weights'], n_simulations=5000, seed=42)

print("=== Monte Carlo результаты (5000 симуляций) ===")
print(f"Средняя годовая доходность:  {mc['annual_return'].mean():.2%}")
print(f"Средняя годовая волатильность: {mc['annual_volatility'].mean():.2%}")
print(f"Средняя максимальная просадка: {mc['max_drawdown'].mean():.2%}")
print(f"Средний Sharpe:               {mc['sharpe'].mean():.3f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(mc['annual_return'] * 100, bins=80, color='#1f77b4', alpha=0.7, edgecolor='white')
axes[0].axvline(x=mc['annual_return'].mean() * 100, color='red', linestyle='--', label='Mean')
axes[0].set_xlabel('Годовая доходность (%)')
axes[0].set_ylabel('Частота')
axes[0].set_title('Распределение доходности')
axes[0].legend()

axes[1].hist(mc['annual_volatility'] * 100, bins=80, color='#ff7f0e', alpha=0.7, edgecolor='white')
axes[1].axvline(x=mc['annual_volatility'].mean() * 100, color='red', linestyle='--', label='Mean')
axes[1].set_xlabel('Годовая волатильность (%)')
axes[1].set_title('Распределение волатильности')
axes[1].legend()

axes[2].hist(mc['max_drawdown'] * 100, bins=80, color='#2ca02c', alpha=0.7, edgecolor='white')
axes[2].axvline(x=mc['max_drawdown'].mean() * 100, color='red', linestyle='--', label='Mean')
axes[2].set_xlabel('Максимальная просадка (%)')
axes[2].set_title('Распределение просадки')
axes[2].legend()

plt.tight_layout()
plt.show()

## 12. Анализ рисков: VaR и CVaR

In [ ]:
var_95 = var_historical(clique_returns, opt['weights'], confidence=0.95)
cvar_95 = cvar_historical(clique_returns, opt['weights'], confidence=0.95)
var_99 = var_historical(clique_returns, opt['weights'], confidence=0.99)
cvar_99 = cvar_historical(clique_returns, opt['weights'], confidence=0.99)

print("=== Анализ рисков (Max Sharpe Portfolio) ===")
print(f"VaR  95%:  {var_95:.2%}  (суточные потери с 5% вероятностью)")
print(f"CVaR 95%:  {cvar_95:.2%}  (средние потери за порогом VaR)")
print(f"VaR  99%:  {var_99:.2%}  (суточные потери с 1% вероятностью)")
print(f"CVaR 99%:  {cvar_99:.2%}  (средние потери за порогом VaR)")

## 13. Бета и альфа акций клики

In [ ]:
market_returns = returns.mean(axis=1)
beta = compute_beta(returns[clique], market_returns)
alpha = compute_alpha(returns[clique], market_returns)

print("=== Бета и альфа акций клики ===")
for t in sorted(clique):
    print(f"  {t}: beta={beta[t]:.3f}, alpha={alpha[t]:.2%}")

## 14. Скользящие корреляции

In [ ]:
roll_corr = rolling_correlation(returns[clique], window=60)

fig, ax = plt.subplots(figsize=(14, 6))
for pair_name, series in list(roll_corr.items())[:10]:
    ax.plot(series.index, series.values, alpha=0.5, label=pair_name)
ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax.set_title('Скользящие корреляции (60-дневное окно, топ-10 пар)')
ax.set_ylabel('Корреляция')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
plt.tight_layout()
plt.show()

## 15. Ребалансирование

In [ ]:
rebal_config = RebalanceConfig(
    target_weights={t: opt['weights'][i] for i, t in enumerate(clique)},
    rebalance_freq_days=REBALANCE_FREQ_DAYS,
    transaction_cost_bps=TRANSACTION_COST_BPS,
)

rebal_result = simulate_rebalancing(clique_returns, rebal_config)
bh_result = simulate_buy_and_hold(clique_returns, rebal_config.target_weights)

print("=== Ребалансирование ===")
print(f"Rebalancing: return={rebal_result.annual_return:.2%}, sharpe={rebal_result.sharpe:.3f}, cost={rebal_result.total_cost:,.0f} RUB")
print(f"Buy & Hold:  return={bh_result.annual_return:.2%}, sharpe={bh_result.sharpe:.3f}")

comp = compare_strategies(clique_returns, rebal_config)
print(comp.to_string(index=False))

## 16. Стресс-тестирование

In [ ]:
stress_results = run_all_scenarios(returns, opt['weights'])
stress_df = stress_results_to_dataframe(stress_results)
print(stress_df.to_string(index=False))

## 17. Сводные метрики

In [ ]:
metrics = portfolio_metrics(
    opt['weights'], mean_ret, cov, returns=clique_returns
)
print("=== Метрики оптимального портфеля ===")
print(f"Годовая доходность:        {metrics['return']:.2%}")
print(f"Годовая волатильность:     {metrics['volatility']:.2%}")
print(f"Sharpe Ratio:              {metrics['sharpe']:.3f}")
print(f"Sortino Ratio:             {metrics['sortino']:.3f}")
print(f"Максимальная просадка:     {metrics['max_drawdown']:.2%}")
if metrics.get('calmar') is not None:
    print(f"Calmar Ratio:              {metrics['calmar']:.3f}")
if metrics.get('information_ratio') is not None:
    print(f"Information Ratio:         {metrics['information_ratio']:.3f}")

## 18. Сохранение профиля портфеля

In [ ]:
profile_path = save_profile(
    name="my_optimal_portfolio",
    clique=clique,
    weights={t: float(w) for t, w in zip(clique, opt['weights'])},
    metrics=metrics,
    params={
        'corr_threshold': CORR_THRESHOLD,
        'min_turnover': MIN_TURNOVER,
        'max_weight': MAX_WEIGHT,
    },
)
print(f"Профиль сохранён: {profile_path}")

# Загрузка
loaded = load_profile("my_optimal_portfolio")
print(f"Загружен профиль: {loaded['name']}")
print(f"Акции: {', '.join(loaded['clique'])}")
print(f"Сохранённые профили: {list_profiles()}")